In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

DATA_DIR = Path("../data")

train = pd.read_csv(DATA_DIR / "train.csv")
validation = pd.read_csv(DATA_DIR / "validation.csv")
test = pd.read_csv(DATA_DIR / "test_input.csv")

# Convert datetime
for df in [train, validation, test]:
    df["datetime"] = pd.to_datetime(
        df["datetime"],
        format="%d-%m-%Y %H:%M"
    )

targets = [
    "nat_demand",
    "load_tocumen_mwh",
    "load_santiago_mwh",
    "load_david_mwh"
]

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (34343, 51)
Validation: (7248, 51)
Test: (2352, 47)


In [2]:
def add_time_features(df):
    df = df.copy()

    # Cyclic hour-of-day features
    df["hour_sin"] = np.sin(2 * np.pi * df["hourOfDay"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hourOfDay"] / 24)

    # Cyclic day-of-week features
    df["dow_sin"] = np.sin(2 * np.pi * (df["dayOfWeek"] - 1) / 7)
    df["dow_cos"] = np.cos(2 * np.pi * (df["dayOfWeek"] - 1) / 7)

    # Seasonal features
    month = df["datetime"].dt.month
    day_of_year = df["datetime"].dt.dayofyear

    df["month_sin"] = np.sin(2 * np.pi * (month - 1) / 12)
    df["month_cos"] = np.cos(2 * np.pi * (month - 1) / 12)

    df["doy_sin"] = np.sin(2 * np.pi * (day_of_year - 1) / 365.25)
    df["doy_cos"] = np.cos(2 * np.pi * (day_of_year - 1) / 365.25)

    # Peak-period indicators based on our EDA
    df["daytime_peak"] = df["hourOfDay"].between(10, 15).astype(int)
    df["evening_peak"] = df["hourOfDay"].between(18, 20).astype(int)

    # Useful interactions
    df["hour_weekend"] = df["hourOfDay"] * df["weekend"]
    df["hour_holiday"] = df["hourOfDay"] * df["holiday"]

    return df


train_fe = add_time_features(train)
validation_fe = add_time_features(validation)
test_fe = add_time_features(test)

print("Original features:", train.shape[1])
print("After time engineering:", train_fe.shape[1])

new_time_features = [
    col for col in train_fe.columns
    if col not in train.columns
]

print("\nNew features:")
print(new_time_features)

Original features: 51
After time engineering: 63

New features:
['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos', 'daytime_peak', 'evening_peak', 'hour_weekend', 'hour_holiday']


In [3]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Complete validation windows only
window_counts = validation_fe.groupby("window_id").size()
complete_windows = window_counts[window_counts == 168].index

val_complete = validation_fe[
    validation_fe["window_id"].isin(complete_windows)
].copy()

# Original usable features
base_features = [
    col for col in test.columns
    if col not in ["datetime", "split", "window_id", "horizon_hour"]
]

# Add engineered time features
time_engineered = [
    "hour_sin", "hour_cos",
    "dow_sin", "dow_cos",
    "month_sin", "month_cos",
    "doy_sin", "doy_cos",
    "daytime_peak", "evening_peak",
    "hour_weekend", "hour_holiday"
]

engineered_features = base_features + time_engineered

results_time = []

for target in targets:
    model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(train_fe[engineered_features], train_fe[target])

    y_true = val_complete[target]
    y_pred = model.predict(val_complete[engineered_features])

    results_time.append({
        "Target": target,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE (%)": np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
        "R²": r2_score(y_true, y_pred)
    })

results_time = pd.DataFrame(results_time)

display(results_time.round(4))

,Target,MAE,RMSE,MAPE (%),R²
0,nat_demand,46.3119,64.9463,4.1090,0.8826
1,load_tocumen_mwh,37.8863,53.2227,4.1031,0.8824
2,load_santiago_mwh,3.0304,4.2163,4.1809,0.8809
3,load_david_mwh,5.5029,7.6739,4.1753,0.8810


In [4]:
def add_demand_features(df):
    df = df.copy()

    node_prefixes = [
        "national",
        "tocumen",
        "santiago",
        "david"
    ]

    for node in node_prefixes:
        w2 = f"{node}_week_X-2_mwh"
        w3 = f"{node}_week_X-3_mwh"
        w4 = f"{node}_week_X-4_mwh"
        ma = f"{node}_MA_X-4_mwh"

        # Week-to-week changes
        df[f"{node}_trend_2_3"] = df[w2] - df[w3]
        df[f"{node}_trend_3_4"] = df[w3] - df[w4]
        df[f"{node}_trend_2_4"] = df[w2] - df[w4]

        # Deviation from moving average
        df[f"{node}_dev_ma"] = df[w2] - df[ma]

        # Relative deviation
        df[f"{node}_ratio_ma"] = df[w2] / (df[ma] + 1e-6)

    return df


train_fe = add_demand_features(train_fe)
validation_fe = add_demand_features(validation_fe)
test_fe = add_demand_features(test_fe)

new_demand_features = [
    col for col in train_fe.columns
    if col not in train.columns
    and col not in time_engineered
]

print("New demand features:", len(new_demand_features))
print(new_demand_features)

New demand features: 20
['national_trend_2_3', 'national_trend_3_4', 'national_trend_2_4', 'national_dev_ma', 'national_ratio_ma', 'tocumen_trend_2_3', 'tocumen_trend_3_4', 'tocumen_trend_2_4', 'tocumen_dev_ma', 'tocumen_ratio_ma', 'santiago_trend_2_3', 'santiago_trend_3_4', 'santiago_trend_2_4', 'santiago_dev_ma', 'santiago_ratio_ma', 'david_trend_2_3', 'david_trend_3_4', 'david_trend_2_4', 'david_dev_ma', 'david_ratio_ma']


In [6]:
# Recreate complete validation set after feature engineering
window_counts = validation_fe.groupby("window_id").size()

complete_windows = window_counts[window_counts == 168].index

val_complete = validation_fe[
    validation_fe["window_id"].isin(complete_windows)
].copy()

print("Validation rows:", len(val_complete))
print("Validation columns:", len(val_complete.columns))

Validation rows: 5712
Validation columns: 83


In [7]:
demand_engineered_features = (
    time_engineered
    + new_demand_features
)

all_features_v2 = base_features + demand_engineered_features

results_demand = []

for target in targets:
    model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        train_fe[all_features_v2],
        train_fe[target]
    )

    y_true = val_complete[target]
    y_pred = model.predict(val_complete[all_features_v2])

    results_demand.append({
        "Target": target,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE (%)": np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
        "R²": r2_score(y_true, y_pred)
    })

results_demand = pd.DataFrame(results_demand)

display(results_demand.round(4))

,Target,MAE,RMSE,MAPE (%),R²
0,nat_demand,45.7769,64.3206,4.0600,0.8849
1,load_tocumen_mwh,37.3695,52.4490,4.0502,0.8858
2,load_santiago_mwh,2.9930,4.1843,4.1234,0.8827
3,load_david_mwh,5.4008,7.5409,4.0987,0.8851


In [8]:
def add_weather_features(df):
    df = df.copy()

    # Temperature interactions
    df["toc_temp_humidity"] = df["T2M_toc"] * df["QV2M_toc"]
    df["san_temp_humidity"] = df["T2M_san"] * df["QV2M_san"]
    df["dav_temp_humidity"] = df["T2M_dav"] * df["QV2M_dav"]

    # Nonlinear temperature effects
    df["toc_temp_sq"] = df["T2M_toc"] ** 2
    df["san_temp_sq"] = df["T2M_san"] ** 2
    df["dav_temp_sq"] = df["T2M_dav"] ** 2

    # Cross-zone temperature differences
    df["temp_toc_san_diff"] = df["T2M_toc"] - df["T2M_san"]
    df["temp_toc_dav_diff"] = df["T2M_toc"] - df["T2M_dav"]
    df["temp_san_dav_diff"] = df["T2M_san"] - df["T2M_dav"]

    return df


train_fe = add_weather_features(train_fe)
validation_fe = add_weather_features(validation_fe)
test_fe = add_weather_features(test_fe)

weather_engineered_features = [
    "toc_temp_humidity", "san_temp_humidity", "dav_temp_humidity",
    "toc_temp_sq", "san_temp_sq", "dav_temp_sq",
    "temp_toc_san_diff", "temp_toc_dav_diff", "temp_san_dav_diff"
]

print("New weather features:", len(weather_engineered_features))
print(weather_engineered_features)

New weather features: 9
['toc_temp_humidity', 'san_temp_humidity', 'dav_temp_humidity', 'toc_temp_sq', 'san_temp_sq', 'dav_temp_sq', 'temp_toc_san_diff', 'temp_toc_dav_diff', 'temp_san_dav_diff']


In [10]:
# Recreate complete validation set after weather feature engineering
window_counts = validation_fe.groupby("window_id").size()
complete_windows = window_counts[window_counts == 168].index

val_complete = validation_fe[
    validation_fe["window_id"].isin(complete_windows)
].copy()

print("Validation rows:", len(val_complete))
print("Validation columns:", len(val_complete.columns))

Validation rows: 5712
Validation columns: 92


In [11]:
weather_added_features = (
    time_engineered
    + new_demand_features
    + weather_engineered_features
)

all_features_v3 = base_features + weather_added_features

results_weather = []

for target in targets:
    model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        train_fe[all_features_v3],
        train_fe[target]
    )

    y_true = val_complete[target]
    y_pred = model.predict(val_complete[all_features_v3])

    results_weather.append({
        "Target": target,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE (%)": np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
        "R²": r2_score(y_true, y_pred)
    })

results_weather = pd.DataFrame(results_weather)

display(results_weather.round(4))

,Target,MAE,RMSE,MAPE (%),R²
0,nat_demand,45.7542,64.3338,4.0585,0.8848
1,load_tocumen_mwh,37.2390,52.4325,4.0381,0.8859
2,load_santiago_mwh,2.9500,4.1207,4.0762,0.8862
3,load_david_mwh,5.3995,7.5561,4.1040,0.8847


In [12]:
def add_renewable_features(df):
    df = df.copy()

    # Solar / wind composition
    df["solar_share_vre"] = (
        df["total_solar_gen_mwh"] /
        (df["total_vre_gen_mwh"] + 1e-6)
    )

    df["wind_share_vre"] = (
        df["total_wind_gen_mwh"] /
        (df["total_vre_gen_mwh"] + 1e-6)
    )

    # Renewable generation relative to historical demand
    df["vre_to_national_ma"] = (
        df["total_vre_gen_mwh"] /
        (df["national_MA_X-4_mwh"] + 1e-6)
    )

    df["solar_to_national_ma"] = (
        df["total_solar_gen_mwh"] /
        (df["national_MA_X-4_mwh"] + 1e-6)
    )

    df["wind_to_national_ma"] = (
        df["total_wind_gen_mwh"] /
        (df["national_MA_X-4_mwh"] + 1e-6)
    )

    # Regional renewable totals
    df["tocumen_vre"] = (
        df["tocumen_wind_gen_mwh"] +
        df["tocumen_solar_gen_mwh"]
    )

    df["santiago_vre"] = (
        df["santiago_wind_gen_mwh"] +
        df["santiago_solar_gen_mwh"]
    )

    df["david_vre"] = (
        df["david_wind_gen_mwh"] +
        df["david_solar_gen_mwh"]
    )

    return df


train_fe = add_renewable_features(train_fe)
validation_fe = add_renewable_features(validation_fe)
test_fe = add_renewable_features(test_fe)

renewable_engineered_features = [
    "solar_share_vre",
    "wind_share_vre",
    "vre_to_national_ma",
    "solar_to_national_ma",
    "wind_to_national_ma",
    "tocumen_vre",
    "santiago_vre",
    "david_vre"
]

print("New renewable features:", len(renewable_engineered_features))
print(renewable_engineered_features)

New renewable features: 8
['solar_share_vre', 'wind_share_vre', 'vre_to_national_ma', 'solar_to_national_ma', 'wind_to_national_ma', 'tocumen_vre', 'santiago_vre', 'david_vre']


In [13]:
window_counts = validation_fe.groupby("window_id").size()
complete_windows = window_counts[window_counts == 168].index

val_complete = validation_fe[
    validation_fe["window_id"].isin(complete_windows)
].copy()

In [14]:
renewable_added_features = (
    time_engineered
    + new_demand_features
    + weather_engineered_features
    + renewable_engineered_features
)

all_features_v4 = base_features + renewable_added_features

results_renewable = []

for target in targets:
    model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(train_fe[all_features_v4], train_fe[target])

    y_true = val_complete[target]
    y_pred = model.predict(val_complete[all_features_v4])

    results_renewable.append({
        "Target": target,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE (%)": np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
        "R²": r2_score(y_true, y_pred)
    })

results_renewable = pd.DataFrame(results_renewable)

display(results_renewable.round(4))

,Target,MAE,RMSE,MAPE (%),R²
0,nat_demand,45.7005,64.3243,4.0566,0.8849
1,load_tocumen_mwh,37.2661,52.5017,4.0420,0.8856
2,load_santiago_mwh,2.9627,4.1471,4.0908,0.8848
3,load_david_mwh,5.3675,7.5463,4.0861,0.8850


In [15]:
def add_cross_node_features(df):
    df = df.copy()

    nodes = ["tocumen", "santiago", "david"]

    # Compare historical moving averages with national demand
    for node in nodes:
        node_ma = f"{node}_MA_X-4_mwh"

        df[f"{node}_national_ma_ratio"] = (
            df[node_ma] / (df["national_MA_X-4_mwh"] + 1e-6)
        )

        df[f"{node}_national_ma_diff"] = (
            df[node_ma] - df["national_MA_X-4_mwh"]
        )

    # Relative weekly changes between zones
    df["tocumen_santiago_ma_ratio"] = (
        df["tocumen_MA_X-4_mwh"] /
        (df["santiago_MA_X-4_mwh"] + 1e-6)
    )

    df["tocumen_david_ma_ratio"] = (
        df["tocumen_MA_X-4_mwh"] /
        (df["david_MA_X-4_mwh"] + 1e-6)
    )

    df["santiago_david_ma_ratio"] = (
        df["santiago_MA_X-4_mwh"] /
        (df["david_MA_X-4_mwh"] + 1e-6)
    )

    return df


train_fe = add_cross_node_features(train_fe)
validation_fe = add_cross_node_features(validation_fe)
test_fe = add_cross_node_features(test_fe)

cross_node_features = [
    "tocumen_national_ma_ratio",
    "tocumen_national_ma_diff",
    "santiago_national_ma_ratio",
    "santiago_national_ma_diff",
    "david_national_ma_ratio",
    "david_national_ma_diff",
    "tocumen_santiago_ma_ratio",
    "tocumen_david_ma_ratio",
    "santiago_david_ma_ratio"
]

print("New cross-node features:", len(cross_node_features))

New cross-node features: 9


In [16]:
cross_node_added_features = (
    time_engineered
    + new_demand_features
    + weather_engineered_features
    + cross_node_features
)

all_features_v5 = base_features + cross_node_added_features

window_counts = validation_fe.groupby("window_id").size()
complete_windows = window_counts[window_counts == 168].index

val_complete = validation_fe[
    validation_fe["window_id"].isin(complete_windows)
].copy()

results_cross = []

for target in targets:
    model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(train_fe[all_features_v5], train_fe[target])

    y_true = val_complete[target]
    y_pred = model.predict(val_complete[all_features_v5])

    results_cross.append({
        "Target": target,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE (%)": np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
        "R²": r2_score(y_true, y_pred)
    })

results_cross = pd.DataFrame(results_cross)

display(results_cross.round(4))

,Target,MAE,RMSE,MAPE (%),R²
0,nat_demand,47.3459,65.8221,4.1719,0.8794
1,load_tocumen_mwh,38.2779,53.5145,4.1262,0.8811
2,load_santiago_mwh,3.0761,4.2536,4.2127,0.8788
3,load_david_mwh,5.5284,7.7157,4.1830,0.8797


In [19]:
current_features = (
    base_features
    + time_engineered
    + new_demand_features
    + weather_engineered_features
)
print("Current features:", len(current_features))


Current features: 84


In [20]:
def add_interaction_features(df):
    df = df.copy()

    # Average temperature across the three zones
    df["avg_temperature"] = (
        df["T2M_toc"] +
        df["T2M_san"] +
        df["T2M_dav"]
    ) / 3

    # Temperature range across zones
    df["temperature_range"] = (
        df[["T2M_toc", "T2M_san", "T2M_dav"]].max(axis=1)
        - df[["T2M_toc", "T2M_san", "T2M_dav"]].min(axis=1)
    )

    # Temperature × hour
    df["temp_hour"] = df["avg_temperature"] * df["hourOfDay"]

    # Temperature × cyclic hour
    df["temp_hour_sin"] = df["avg_temperature"] * df["hour_sin"]
    df["temp_hour_cos"] = df["avg_temperature"] * df["hour_cos"]

    # Temperature × calendar effects
    df["temp_weekend"] = df["avg_temperature"] * df["weekend"]
    df["temp_holiday"] = df["avg_temperature"] * df["holiday"]

    # Solar × hour
    df["solar_hour"] = (
        df["total_solar_gen_mwh"] * df["hourOfDay"]
    )

    # Solar × cyclic hour
    df["solar_hour_sin"] = (
        df["total_solar_gen_mwh"] * df["hour_sin"]
    )
    df["solar_hour_cos"] = (
        df["total_solar_gen_mwh"] * df["hour_cos"]
    )

    # Solar × peak periods
    df["solar_daytime_peak"] = (
        df["total_solar_gen_mwh"] * df["daytime_peak"]
    )

    df["solar_evening_peak"] = (
        df["total_solar_gen_mwh"] * df["evening_peak"]
    )

    return df


train_fe = add_interaction_features(train_fe)
validation_fe = add_interaction_features(validation_fe)
test_fe = add_interaction_features(test_fe)

interaction_features = [
    "avg_temperature",
    "temperature_range",
    "temp_hour",
    "temp_hour_sin",
    "temp_hour_cos",
    "temp_weekend",
    "temp_holiday",
    "solar_hour",
    "solar_hour_sin",
    "solar_hour_cos",
    "solar_daytime_peak",
    "solar_evening_peak"
]

print("New interaction features:", len(interaction_features))
print(interaction_features)

New interaction features: 12
['avg_temperature', 'temperature_range', 'temp_hour', 'temp_hour_sin', 'temp_hour_cos', 'temp_weekend', 'temp_holiday', 'solar_hour', 'solar_hour_sin', 'solar_hour_cos', 'solar_daytime_peak', 'solar_evening_peak']


In [21]:
window_counts = validation_fe.groupby("window_id").size()
complete_windows = window_counts[window_counts == 168].index

val_complete = validation_fe[
    validation_fe["window_id"].isin(complete_windows)
].copy()

In [22]:
interaction_added_features = current_features + interaction_features

results_interaction = []

for target in targets:
    model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        train_fe[interaction_added_features],
        train_fe[target]
    )

    y_true = val_complete[target]
    y_pred = model.predict(
        val_complete[interaction_added_features]
    )

    results_interaction.append({
        "Target": target,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE (%)": np.mean(
            np.abs((y_true - y_pred) / y_true)
        ) * 100,
        "R²": r2_score(y_true, y_pred)
    })

results_interaction = pd.DataFrame(results_interaction)

display(results_interaction.round(4))

,Target,MAE,RMSE,MAPE (%),R²
0,nat_demand,46.3703,64.7913,4.1018,0.8832
1,load_tocumen_mwh,37.5125,52.6460,4.0565,0.8850
2,load_santiago_mwh,2.9942,4.1700,4.1208,0.8835
3,load_david_mwh,5.3875,7.5456,4.0909,0.8850


In [23]:
def add_stability_features(df):
    df = df.copy()

    nodes = ["national", "tocumen", "santiago", "david"]

    for node in nodes:
        w2 = f"{node}_week_X-2_mwh"
        w3 = f"{node}_week_X-3_mwh"
        w4 = f"{node}_week_X-4_mwh"

        weekly = df[[w2, w3, w4]]

        # Average of available weekly lags
        df[f"{node}_weekly_mean"] = weekly.mean(axis=1)

        # Historical variability
        df[f"{node}_weekly_std"] = weekly.std(axis=1)
        df[f"{node}_weekly_range"] = weekly.max(axis=1) - weekly.min(axis=1)

        # Relative variability
        df[f"{node}_weekly_cv"] = (
            df[f"{node}_weekly_std"] /
            (df[f"{node}_weekly_mean"] + 1e-6)
        )

        # Direction consistency
        df[f"{node}_trend_consistency"] = (
            np.sign(df[w2] - df[w3]) +
            np.sign(df[w3] - df[w4])
        )

    return df


train_fe = add_stability_features(train_fe)
validation_fe = add_stability_features(validation_fe)
test_fe = add_stability_features(test_fe)

stability_features = [
    col for col in train_fe.columns
    if col.endswith((
        "_weekly_mean",
        "_weekly_std",
        "_weekly_range",
        "_weekly_cv",
        "_trend_consistency"
    ))
]

print("New stability features:", len(stability_features))
print(stability_features)

New stability features: 20
['national_weekly_mean', 'national_weekly_std', 'national_weekly_range', 'national_weekly_cv', 'national_trend_consistency', 'tocumen_weekly_mean', 'tocumen_weekly_std', 'tocumen_weekly_range', 'tocumen_weekly_cv', 'tocumen_trend_consistency', 'santiago_weekly_mean', 'santiago_weekly_std', 'santiago_weekly_range', 'santiago_weekly_cv', 'santiago_trend_consistency', 'david_weekly_mean', 'david_weekly_std', 'david_weekly_range', 'david_weekly_cv', 'david_trend_consistency']


In [24]:
stability_features_added = current_features + stability_features

window_counts = validation_fe.groupby("window_id").size()
complete_windows = window_counts[window_counts == 168].index

val_complete = validation_fe[
    validation_fe["window_id"].isin(complete_windows)
].copy()

results_stability = []

for target in targets:
    model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        train_fe[stability_features_added],
        train_fe[target]
    )

    y_true = val_complete[target]
    y_pred = model.predict(
        val_complete[stability_features_added]
    )

    results_stability.append({
        "Target": target,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE (%)": np.mean(
            np.abs((y_true - y_pred) / y_true)
        ) * 100,
        "R²": r2_score(y_true, y_pred)
    })

results_stability = pd.DataFrame(results_stability)

display(results_stability.round(4))

,Target,MAE,RMSE,MAPE (%),R²
0,nat_demand,45.8096,64.4089,4.0632,0.8845
1,load_tocumen_mwh,37.4441,52.5691,4.0576,0.8853
2,load_santiago_mwh,2.9425,4.1114,4.0616,0.8867
3,load_david_mwh,5.3052,7.4789,4.0413,0.8870


In [25]:
current_features = (
    base_features
    + time_engineered
    + new_demand_features
    + weather_engineered_features
    + stability_features
)

In [26]:
def add_net_load_features(df):
    df = df.copy()

    # Historical demand baseline minus current renewable generation
    df["net_load_ma_proxy"] = (
        df["national_MA_X-4_mwh"]
        - df["total_vre_gen_mwh"]
    )

    df["net_load_w2_proxy"] = (
        df["national_week_X-2_mwh"]
        - df["total_vre_gen_mwh"]
    )

    df["net_load_w3_proxy"] = (
        df["national_week_X-3_mwh"]
        - df["total_vre_gen_mwh"]
    )

    df["net_load_w4_proxy"] = (
        df["national_week_X-4_mwh"]
        - df["total_vre_gen_mwh"]
    )

    # Renewable penetration relative to historical demand
    df["vre_penetration_proxy"] = (
        df["total_vre_gen_mwh"]
        / (df["national_MA_X-4_mwh"] + 1e-6)
    )

    # Solar vs wind composition
    df["solar_wind_balance"] = (
        df["total_solar_gen_mwh"]
        - df["total_wind_gen_mwh"]
    )

    return df


train_fe = add_net_load_features(train_fe)
validation_fe = add_net_load_features(validation_fe)
test_fe = add_net_load_features(test_fe)

net_load_features = [
    "net_load_ma_proxy",
    "net_load_w2_proxy",
    "net_load_w3_proxy",
    "net_load_w4_proxy",
    "vre_penetration_proxy",
    "solar_wind_balance"
]

print("New net-load features:", len(net_load_features))

New net-load features: 6


In [27]:
net_load_feature_set = current_features + net_load_features

window_counts = validation_fe.groupby("window_id").size()
complete_windows = window_counts[window_counts == 168].index

val_complete = validation_fe[
    validation_fe["window_id"].isin(complete_windows)
].copy()

results_net_load = []

for target in targets:
    model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(train_fe[net_load_feature_set], train_fe[target])

    y_true = val_complete[target]
    y_pred = model.predict(val_complete[net_load_feature_set])

    results_net_load.append({
        "Target": target,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE (%)": np.mean(
            np.abs((y_true - y_pred) / y_true)
        ) * 100,
        "R²": r2_score(y_true, y_pred)
    })

results_net_load = pd.DataFrame(results_net_load)

display(results_net_load.round(4))

,Target,MAE,RMSE,MAPE (%),R²
0,nat_demand,45.4564,63.8399,4.0316,0.8866
1,load_tocumen_mwh,37.0656,52.0682,4.0220,0.8875
2,load_santiago_mwh,2.9409,4.1086,4.0612,0.8869
3,load_david_mwh,5.3644,7.5137,4.0815,0.8859


In [28]:
current_features = (
    base_features
    + time_engineered
    + new_demand_features
    + weather_engineered_features
    + stability_features
    + net_load_features
)

print("Current feature count:", len(current_features))

Current feature count: 110


In [29]:
from xgboost import XGBRegressor

feature_importance = {}

for target in targets:
    model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(train_fe[current_features], train_fe[target])

    importance_df = pd.DataFrame({
        "feature": current_features,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)

    feature_importance[target] = importance_df

    print(f"\n===== {target} =====")
    display(importance_df.head(15).round(4))


===== nat_demand =====


,feature,importance
29,santiago_MA_X-4_mwh,0.3583
33,david_MA_X-4_mwh,0.2148
25,tocumen_MA_X-4_mwh,0.1383
21,national_MA_X-4_mwh,0.1120
51,daytime_peak,0.0529
3,holiday,0.0378
96,santiago_weekly_range,0.0041
4,Holiday_ID,0.0038
86,national_weekly_range,0.0037
6,T2M_toc,0.0030



===== load_tocumen_mwh =====


,feature,importance
29,santiago_MA_X-4_mwh,0.3408
33,david_MA_X-4_mwh,0.2228
25,tocumen_MA_X-4_mwh,0.1545
21,national_MA_X-4_mwh,0.1196
3,holiday,0.0390
51,daytime_peak,0.0388
86,national_weekly_range,0.0047
4,Holiday_ID,0.0040
96,santiago_weekly_range,0.0034
78,toc_temp_sq,0.0033



===== load_santiago_mwh =====


,feature,importance
29,santiago_MA_X-4_mwh,0.4211
33,david_MA_X-4_mwh,0.2196
25,tocumen_MA_X-4_mwh,0.1489
51,daytime_peak,0.0451
3,holiday,0.0423
21,national_MA_X-4_mwh,0.0140
79,san_temp_sq,0.0097
10,T2M_san,0.0046
4,Holiday_ID,0.0045
96,santiago_weekly_range,0.0041



===== load_david_mwh =====


,feature,importance
29,santiago_MA_X-4_mwh,0.4034
33,david_MA_X-4_mwh,0.2690
25,tocumen_MA_X-4_mwh,0.1345
51,daytime_peak,0.0444
3,holiday,0.0414
80,dav_temp_sq,0.0073
14,T2M_dav,0.0054
96,santiago_weekly_range,0.0049
41,total_solar_gen_mwh,0.0048
4,Holiday_ID,0.0044


In [30]:
# Combine feature importance across all four targets
importance_all = pd.DataFrame(index=current_features)

for target in targets:
    imp = pd.Series(
        feature_importance[target]["importance"].values,
        index=feature_importance[target]["feature"]
    )
    imp = imp / imp.sum()  # normalize
    importance_all[target] = imp

importance_all["mean_importance"] = importance_all.mean(axis=1)

importance_ranked = (
    importance_all["mean_importance"]
    .sort_values(ascending=False)
)

display(importance_ranked.head(20).to_frame("mean_importance").round(5))

,mean_importance
santiago_MA_X-4_mwh,0.38091
david_MA_X-4_mwh,0.23155
tocumen_MA_X-4_mwh,0.14405
national_MA_X-4_mwh,0.06184
daytime_peak,0.04529
holiday,0.04014
Holiday_ID,0.00418
santiago_weekly_range,0.00414
san_temp_sq,0.00368
national_weekly_range,0.00354


In [31]:
top_80_features = importance_ranked.head(80).index.tolist()
top_50_features = importance_ranked.head(50).index.tolist()

print("110 features:", len(current_features))
print("Top 80:", len(top_80_features))
print("Top 50:", len(top_50_features))

110 features: 110
Top 80: 80
Top 50: 50


In [32]:
feature_sets = {
    "110_features": current_features,
    "80_features": top_80_features,
    "50_features": top_50_features
}

selection_results = []

for set_name, feature_set in feature_sets.items():

    print(f"\nTraining {set_name}...")

    for target in targets:

        model = XGBRegressor(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            train_fe[feature_set],
            train_fe[target]
        )

        y_true = val_complete[target]
        y_pred = model.predict(
            val_complete[feature_set]
        )

        selection_results.append({
            "Feature_Set": set_name,
            "Target": target,
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "MAPE (%)": np.mean(
                np.abs((y_true - y_pred) / y_true)
            ) * 100,
            "R²": r2_score(y_true, y_pred)
        })

selection_results = pd.DataFrame(selection_results)

display(selection_results.round(4))


Training 110_features...

Training 80_features...

Training 50_features...


,Feature_Set,Target,MAE,RMSE,MAPE (%),R²
0,110_features,nat_demand,45.4564,63.8399,4.0316,0.8866
1,110_features,load_tocumen_mwh,37.0656,52.0682,4.0220,0.8875
2,110_features,load_santiago_mwh,2.9409,4.1086,4.0612,0.8869
3,110_features,load_david_mwh,5.3644,7.5137,4.0815,0.8859
4,80_features,nat_demand,46.0779,64.6337,4.0870,0.8837
5,80_features,load_tocumen_mwh,37.3065,52.4935,4.0445,0.8856
6,80_features,load_santiago_mwh,2.9774,4.1735,4.1131,0.8833
7,80_features,load_david_mwh,5.3189,7.4928,4.0532,0.8866
8,50_features,nat_demand,46.8085,65.6914,4.1405,0.8799
9,50_features,load_tocumen_mwh,38.1555,53.7783,4.1243,0.8800


## Conclusion

In this notebook, we engineered and tested several groups of features instead of adding them blindly. Time, demand trend, weather, stability, and net load proxy features showed useful improvements over the original XGBoost baseline, while the tested renewable derived, cross node, and interaction features did not provide consistent improvements.

The strongest model so far uses 110 features and achieved an average MAPE of about 4.05% and average R² of about 0.887 on the 34 complete validation windows. Feature importance analysis also showed that historical demand patterns are the main source of predictive signal, with calendar, weather and renewable variables providing additional information.

We will carry the 110 feature set forward and test stronger model architectures and optimization strategies next.